# Neural Network in JAX

This notebook demonstrates a supervised learning workflow using a small multilayer perceptron.

The example learns a nonlinear XOR-style classification problem.

## Theory

A neural network combines linear layers and nonlinear activations. For one hidden layer, the forward pass is:

$$
h = 	anh(XW_1 + b_1), uad at{y} = igma(hW_2 + b_2)
$$

The network is trained by minimizing binary cross-entropy.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

key = jax.random.PRNGKey(3)
keys = jax.random.split(key, 4)
blobs = [
    jax.random.normal(keys[0], (80, 2)) * 0.25 + jnp.array([-1.0, -1.0]),
    jax.random.normal(keys[1], (80, 2)) * 0.25 + jnp.array([1.0, 1.0]),
    jax.random.normal(keys[2], (80, 2)) * 0.25 + jnp.array([-1.0, 1.0]),
    jax.random.normal(keys[3], (80, 2)) * 0.25 + jnp.array([1.0, -1.0])
]
x = jnp.concatenate(blobs, axis=0)
y = jnp.concatenate([jnp.zeros((160, 1)), jnp.ones((160, 1))], axis=0)
perm = jax.random.permutation(key, x.shape[0])
x = x[perm]
y = y[perm]

In [ ]:
def init_params(key, hidden_size=8):
    k1, k2 = jax.random.split(key)
    return {
        'w1': jax.random.normal(k1, (2, hidden_size)) * 0.4,
        'b1': jnp.zeros((hidden_size,)),
        'w2': jax.random.normal(k2, (hidden_size, 1)) * 0.4,
        'b2': jnp.zeros((1,))
    }

def forward(params, inputs):
    hidden = jnp.tanh(inputs @ params['w1'] + params['b1'])
    logits = hidden @ params['w2'] + params['b2']
    return jax.nn.sigmoid(logits)

def loss_fn(params, inputs, targets):
    predictions = jnp.clip(forward(params, inputs), 1e-7, 1.0 - 1e-7)
    return -jnp.mean(targets * jnp.log(predictions) + (1.0 - targets) * jnp.log(1.0 - predictions))

def accuracy(params, inputs, targets):
    predictions = (forward(params, inputs) >= 0.5).astype(jnp.float32)
    return jnp.mean(predictions == targets)

def train(inputs, targets, learning_rate=0.1, steps=500):
    params = init_params(jax.random.PRNGKey(7))
    grad_fn = jax.grad(loss_fn)
    history = []
    for step in range(steps):
        gradients = grad_fn(params, inputs, targets)
        params = jax.tree_util.tree_map(lambda p, g: p - learning_rate * g, params, gradients)
        if step % 100 == 0 or step == steps - 1:
            history.append((step, float(loss_fn(params, inputs, targets)), float(accuracy(params, inputs, targets))))
    return params, history

params, history = train(x, y)
print(history)
print(f'final accuracy = {float(accuracy(params, x, y)):.3f}')

## Result

The trained network should separate the nonlinear classes with a curved decision boundary.

In [ ]:
grid_x, grid_y = jnp.meshgrid(jnp.linspace(-2.0, 2.0, 200), jnp.linspace(-2.0, 2.0, 200))
grid_points = jnp.stack([grid_x.ravel(), grid_y.ravel()], axis=1)
grid_pred = forward(params, grid_points).reshape(grid_x.shape)

plt.figure(figsize=(7, 6))
plt.contourf(grid_x, grid_y, grid_pred, levels=20, cmap='RdBu', alpha=0.5)
plt.scatter(x[:, 0], x[:, 1], c=y[:, 0], cmap='bwr', edgecolor='black', s=25)
plt.title('Neural Network in JAX')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()